# Decks and play

What the decks look like and how the cards get played: the mana curve of the decks by mode and pilot, how often
a card that reached a hand is played, the cards most often drawn and not played, the turn a card is first
played on, and how the first player and the length of a game differ by mode and patch.

It reads the same game records as the card win rates (SPEC §9.11, R376–R378) and the catalog for each card's
printed cost. A record holds each deck once per game, so a card a deck names twice is one card in the curve
(`analysis/jackioh_records`). Nothing here is a rule; the figures are counts and shares over the games read.

**Inputs** come from the environment, and default to the committed fixtures (`analysis/README.md`):

| Variable | What | Default |
|---|---|---|
| `JACKIOH_RECORDS` | record files, comma-separated | `analysis/fixtures/dev-40.jsonl`, `live-40.jsonl` |
| `JACKIOH_CATALOG` | the catalog | `crates/cards/catalog.json` |
| `JACKIOH_SOURCE` | `live` (the default), `dev` or `all`: the games read, as a query of R378's does | `live` |
| `JACKIOH_PATCH` | the patch for the decks and the cards (the last section reads every patch) | the newest patch in the records |
| `JACKIOH_MIN_SAMPLE` | hands a card needs to be ranked by its play rate | the server's `CARD_STATS_MIN_SAMPLE` |

In [ ]:
import os
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import PercentFormatter


def find_analysis():
    """The `analysis/` directory, whether Jupyter started here, at the repository root or anywhere below it."""
    for here in (Path.cwd(), *Path.cwd().parents):
        for candidate in (here, here / "analysis"):
            if (candidate / "jackioh_records" / "__init__.py").is_file():
                return candidate
    raise RuntimeError("Open this notebook from inside the JackiOh repository")


ANALYSIS = find_analysis()
sys.path.insert(0, str(ANALYSIS))
FIXTURES = ANALYSIS / "fixtures"


def from_repo(path):
    """A path as given: a relative one is read from the repository root, wherever Jupyter started."""
    path = Path(path).expanduser()
    return path if path.is_absolute() else ANALYSIS.parent / path


def paths_from(name, default):
    """A comma-separated list of files from the environment, or the default."""
    raw = os.environ.get(name)
    return [from_repo(part.strip()) for part in raw.split(",") if part.strip()] if raw else default


from jackioh_records import CATALOG_PATH, load, load_catalog

RECORDS = paths_from("JACKIOH_RECORDS", [FIXTURES / "dev-40.jsonl", FIXTURES / "live-40.jsonl"])
CATALOG = from_repo(os.environ.get("JACKIOH_CATALOG", CATALOG_PATH))


def server_min_sample():
    """The server's `CARD_STATS_MIN_SAMPLE`, read off its config so this notebook states no number of its own."""
    config = ANALYSIS.parent / "crates" / "server" / "src" / "config.rs"
    found = re.search(r"CARD_STATS_MIN_SAMPLE: i64 = (\d+);", config.read_text(encoding="utf-8"))
    if found is None:
        raise RuntimeError(f"No CARD_STATS_MIN_SAMPLE in {config}: set JACKIOH_MIN_SAMPLE")
    return int(found[1])


MIN_SAMPLE = int(os.environ.get("JACKIOH_MIN_SAMPLE") or server_min_sample())

print("records:", *RECORDS, sep="\n  ")
print("catalog:", CATALOG)
print("a card needs", MIN_SAMPLE, "hands to be ranked by its play rate")
if any(FIXTURES in path.resolve().parents for path in RECORDS):
    print("These are the committed fixtures, not live data: set JACKIOH_RECORDS (analysis/README.md).")

In [ ]:
plt.rcParams.update(
    {
        "figure.dpi": 100,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": "#8a8a85",
        "axes.labelcolor": "#52514e",
        "axes.titlecolor": "#0b0b0b",
        "axes.titlesize": 11,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "xtick.color": "#52514e",
        "ytick.color": "#52514e",
        "axes.grid": True,
        "axes.axisbelow": True,
        "grid.color": "#e3e2dc",
        "grid.linewidth": 0.6,
        "legend.frameon": False,
    }
)
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK = "#0b0b0b"
MUTED = "#52514e"


def hbars(ax, labels, values, notes, color, xlabel, title, percent=False):
    """Horizontal bars, first row on top, each with its note (the games behind it) printed at its end."""
    rows = range(len(labels))
    ax.barh(rows, values, color=color, height=0.6)
    ax.set_yticks(rows, labels)
    ax.invert_yaxis()
    ax.grid(axis="y", visible=False)
    ax.axvline(0, color="#8a8a85", linewidth=0.8)
    for row, (value, note) in enumerate(zip(values, notes, strict=True)):
        ax.annotate(
            note,
            (value, row),
            xytext=(4 if value >= 0 else -4, 0),
            textcoords="offset points",
            va="center",
            ha="left" if value >= 0 else "right",
            fontsize=8,
            color=INK,
        )
    low, high = ax.get_xlim()
    span = high - low
    ax.set_xlim(low - 0.28 * span if low < 0 else low, high + 0.28 * span)
    if percent:
        ax.xaxis.set_major_formatter(PercentFormatter(1.0))
    ax.set_xlabel(xlabel)
    ax.set_title(title)

In [ ]:
records = load(RECORDS)
catalog = load_catalog(CATALOG)
games = records.games
if games.empty:
    raise ValueError("No game records were read: set JACKIOH_RECORDS")


def version_key(patch):
    """A patch's order: its numbers, then its revision letter (v0.2.10 before v0.2.10b before v0.2.10c)."""
    return tuple(int(part) for part in re.findall(r"\d+", patch)), patch.lstrip("v").lstrip("0123456789.")


# `live` (the default) reads the games players played, `dev` the AI's development runs and `all` both (R378).
SOURCE = os.environ.get("JACKIOH_SOURCE") or "live"
if SOURCE not in ("live", "dev", "all"):
    raise ValueError(f"JACKIOH_SOURCE is live, dev or all, not {SOURCE!r}")
read = games if SOURCE == "all" else games[games["source"] == SOURCE]
if read.empty:
    raise ValueError(f"No {SOURCE} game in the records: set JACKIOH_SOURCE to dev or all, or JACKIOH_RECORDS")
ALL_PATCHES = sorted(games["patch"].unique(), key=version_key)
PATCHES = sorted(read["patch"].unique(), key=version_key)
PATCH = os.environ.get("JACKIOH_PATCH") or PATCHES[-1]
if PATCH not in PATCHES:
    raise ValueError(f"No game of patch {PATCH} in the records (they hold {', '.join(PATCHES)})")

# One row per game, seat and distinct card, with the game's source, mode and patch and the seat's pilot,
# for the games of the patch read.
seats = records.seats.merge(games[["id", "source", "mode", "patch", "turns"]], left_on="game", right_on="id")
cards = records.seat_cards.merge(
    seats[["game", "seat", "pilot", "went_first", "source", "mode", "patch"]], on=["game", "seat"]
)
cards = cards[(cards["patch"] == PATCH) & (cards["source"] == SOURCE if SOURCE != "all" else True)].copy()
cards = cards.merge(catalog[["name", "cost"]], left_on="card", right_index=True, how="left")
cards["name"] = cards["name"].fillna(cards["card"])
# A card's hand: it was in the opening hand or drawn later.
cards["in_hand"] = cards["in_opening"] | cards["drawn"]
decks = seats[(seats["patch"] == PATCH) & (seats["source"] == SOURCE if SOURCE != "all" else True)]
print(f"{SOURCE} games of patch {PATCH}: {decks['game'].nunique()} games, {len(decks)} decks, {cards['card'].nunique()} different cards")
decks.groupby(["mode", "pilot"]).size().rename("decks").to_frame()

## The mana curve

How many cards of each printed cost a deck holds, on average, by mode and pilot (a mode's panel, a pilot's
bars): a card with a variable cost (`X`) is its own column, a card with an Embiggen price is read at its
base price, and costs of 7 and over share the `7+` column. The number of decks is printed in the legend.

In [ ]:
CURVE_TOP = 7  # costs from here up share a column


def printed_cost(cost):
    """A card's cost as the column it belongs in: "0" to "6", "7+", "X" (variable) or "?" (not a cost)."""
    if isinstance(cost, dict):  # a card with an Embiggen price: its base price
        return printed_cost(cost["base"])
    if isinstance(cost, str):
        return cost
    if pd.isna(cost):
        return "?"
    return f"{CURVE_TOP}+" if int(cost) >= CURVE_TOP else str(int(cost))


cards["printed_cost"] = cards["cost"].map(printed_cost)
columns = (
    [str(c) for c in range(CURVE_TOP)]
    + [f"{CURVE_TOP}+"]
    + [c for c in ("X", "?") if (cards["printed_cost"] == c).any()]
)

deck_counts = decks.groupby(["mode", "pilot"]).size()
curve = (
    cards.groupby(["mode", "pilot", "printed_cost"])
    .size()
    .unstack("printed_cost", fill_value=0)
    .reindex(columns=columns, fill_value=0)
)
curve = curve.div(deck_counts.reindex(curve.index), axis=0)

modes = sorted(curve.index.get_level_values("mode").unique())
fig, axes = plt.subplots(1, len(modes), figsize=(4.4 * len(modes), 3.6), sharey=True, squeeze=False)
PILOT_COLOR = {"human": BLUE, "ai": ORANGE}
for ax, mode in zip(axes[0], modes, strict=True):
    pilots = sorted(curve.loc[mode].index)
    width = 0.8 / len(pilots)
    for k, pilot in enumerate(pilots):
        ax.bar(
            [i + (k - (len(pilots) - 1) / 2) * width for i in range(len(columns))],
            curve.loc[(mode, pilot)],
            width=width * 0.9,
            color=PILOT_COLOR[pilot],
            label=f"{pilot} ({deck_counts[(mode, pilot)]} decks)",
        )
    ax.set_xticks(range(len(columns)), columns)
    ax.grid(axis="x", visible=False)
    ax.set_xlabel("printed cost")
    ax.set_title(mode)
    ax.legend(fontsize=8)
axes[0][0].set_ylabel("cards per deck")
fig.tight_layout()
plt.show()

curve.round(2)

## Play rate

A card's play rate is the share of the hands it reached (its opening hand, or drawn later) that played it at
least once. The table has every card with its hands beside it. The first chart is the cards that were left
unplayed in the most hands, as a count: what the AI or the players drew and could not or would not play. The
second is the lowest play rates among cards that reached at least the minimum number of hands.

In [ ]:
hands = cards[cards["in_hand"]]
per_card = (
    hands.groupby(["card", "name", "printed_cost"])
    .agg(hands=("played", "size"), played=("played", "sum"))
    .reset_index()
)
per_card["unplayed"] = per_card["hands"] - per_card["played"]
per_card["play_rate"] = per_card["played"] / per_card["hands"]
per_card["enough"] = per_card["hands"] >= MIN_SAMPLE
print(f"{len(per_card)} cards reached a hand; {int(per_card['enough'].sum())} of them in at least {MIN_SAMPLE} hands")

table = per_card.assign(
    name=per_card["name"] + per_card["enough"].map({True: "", False: " *"}),
    rate=[f"{r:.0%} of {n}" for r, n in zip(per_card["play_rate"], per_card["hands"], strict=True)],
).sort_values(["unplayed", "card"], ascending=[False, True])
print("* fewer than", MIN_SAMPLE, "hands: shown, never ranked by play rate")
table[["card", "name", "printed_cost", "rate", "unplayed"]].set_index("card").head(30)

In [ ]:
def show_unplayed(per_card, top=15):
    ranked = per_card[per_card["unplayed"] > 0].sort_values(["unplayed", "card"], ascending=[False, True]).head(top)
    if ranked.empty:
        print("Every card that reached a hand was played")
        return
    fig, ax = plt.subplots(figsize=(9, 1.2 + 0.32 * len(ranked)))
    hbars(
        ax,
        list(ranked["name"]),
        list(ranked["unplayed"]),
        [f"{row.unplayed} of {row.hands} hands" for row in ranked.itertuples()],
        ORANGE,
        "hands in which the card was drawn and never played",
        "Cards most often drawn and not played",
    )
    fig.tight_layout()
    plt.show()


def show_lowest_play_rates(per_card, top=15):
    pool = per_card[per_card["enough"]].sort_values(["play_rate", "card"])
    if pool.empty:
        print(f"No card has reached {MIN_SAMPLE} hands yet, so none is ranked by its play rate")
        return
    lowest = pool.head(top)
    fig, ax = plt.subplots(figsize=(9, 1.2 + 0.32 * len(lowest)))
    hbars(
        ax,
        list(lowest["name"]),
        list(lowest["play_rate"]),
        [f"{row.play_rate:.0%} · {row.played} of {row.hands} hands" for row in lowest.itertuples()],
        BLUE,
        "share of the hands that played the card",
        f"The {len(lowest)} lowest play rates",
        percent=True,
    )
    fig.tight_layout()
    plt.show()


show_unplayed(per_card)
show_lowest_play_rates(per_card)

## The turn a card is first played

The turn is a seat's own: its first turn is 1 whichever seat it is (the engine counts both seats' turns in
one sequence, so a seat's own turn is the half of it rounded up). Only records that carry `playedTurns` have a
turn; an older file's cards have none and are left out. The first chart is every first play by turn, the
second the median turn of first play by printed cost, with the number of first plays beside each point.

In [ ]:
first = cards[cards["first_played_turn"].notna()].copy()
first["own_turn"] = (first["first_played_turn"].astype("int64") + 1) // 2
print(f"{len(first)} first plays by {first['card'].nunique()} cards")

if first.empty:
    print("No record in these files says which turn a card was played on")
else:
    by_turn = first.groupby("own_turn").size()
    by_cost = first.groupby("printed_cost").agg(median=("own_turn", "median"), plays=("own_turn", "size"))
    by_cost = by_cost.reindex([c for c in columns if c in by_cost.index])

    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    axes[0].bar(by_turn.index, by_turn.values, color=BLUE, width=0.7)
    for turn, count in by_turn.items():
        axes[0].annotate(
            str(count), (turn, count), xytext=(0, 3), textcoords="offset points", ha="center", fontsize=8, color=INK
        )
    axes[0].set_xlabel("the seat's own turn")
    axes[0].set_ylabel("cards first played")
    axes[0].set_title("First plays by turn")
    axes[0].grid(axis="x", visible=False)

    positions = range(len(by_cost))
    axes[1].bar(positions, by_cost["median"], color=BLUE, width=0.7)
    for position, (median, plays) in enumerate(zip(by_cost["median"], by_cost["plays"], strict=True)):
        axes[1].annotate(
            f"{median:g} · {plays}",
            (position, median),
            xytext=(0, 3),
            textcoords="offset points",
            ha="center",
            fontsize=8,
            color=INK,
        )
    axes[1].set_xticks(positions, list(by_cost.index))
    axes[1].grid(axis="x", visible=False)
    axes[1].set_xlabel("printed cost")
    axes[1].set_ylabel("median own turn of first play")
    axes[1].set_title("Median turn of first play by cost (turn · first plays)")
    axes[1].margins(y=0.15)
    fig.tight_layout()
    plt.show()

The 20 cards whose first play came latest, by median turn, with the number of first plays it rests on (a card
played once is one first play: read the count before the turn).

In [ ]:
if first.empty:
    cards_by_turn = None
else:
    cards_by_turn = (
        first.groupby(["card", "name"])
        .agg(first_plays=("own_turn", "size"), median_turn=("own_turn", "median"))
        .reset_index()
        .sort_values(["median_turn", "card"], ascending=[False, True])
        .set_index("card")
        .head(20)
    )
cards_by_turn

## The first player and the length of a game

By patch and mode (every patch, not only the one above): how many games, the share the first player won, the
share the second player won, the share drawn, and the game's length in turns (`turns`, both seats' turns
counted). Development runs and live games are listed apart, since the AI's games are not the players'.

In [ ]:
played = games.assign(
    first_won=games["winner"] == games["first"],
    second_won=(games["winner"] != games["first"]) & (games["winner"] != "draw"),
    drawn=games["winner"] == "draw",
)
by_group = played.groupby(["patch", "mode", "source"]).agg(
    games=("id", "size"),
    first_won=("first_won", "sum"),
    second_won=("second_won", "sum"),
    drawn=("drawn", "sum"),
    mean_turns=("turns", "mean"),
    median_turns=("turns", "median"),
)
by_group["first_win_rate"] = by_group["first_won"] / by_group["games"]
by_group = by_group.sort_index(key=lambda index: index.map(ALL_PATCHES.index) if index.name == "patch" else index)
by_group.round(3)

In [ ]:
def group_label(key):
    patch, mode, source = key
    return f"{patch} · {mode} · {source}"


labels = [group_label(key) for key in by_group.index]
fig, axes = plt.subplots(1, 2, figsize=(14, 1.4 + 0.4 * len(labels)))
hbars(
    axes[0],
    labels,
    list(by_group["first_win_rate"]),
    [f"{row.first_win_rate:.0%} · {row.games} games" for row in by_group.itertuples()],
    BLUE,
    "share of games the first player won (a draw is not a win)",
    "First player's win rate",
    percent=True,
)
hbars(
    axes[1],
    labels,
    list(by_group["mean_turns"]),
    [f"{row.mean_turns:.1f} · {row.games} games" for row in by_group.itertuples()],
    AQUA,
    "turns, both seats counted",
    "Mean length of a game",
)
fig.tight_layout()
plt.show()